# 🎓 Learning Resource Finder — multi-agent demo
Four LlamaIndex agents (Info → Search ⇄ Evaluation → Learning Path) that reason, call tools and hand off to each other.

**In Colab:** clone the repo, `pip install -r requirements.txt`, then run from the repo root (see README).

In [ ]:
# !pip install -q -r requirements.txt   # uncomment on first run

In [ ]:
import os, sys, getpass, pathlib

def find_root():
    """Find the folder that contains the `learning_agent` package (works locally and in Colab)."""
    here = pathlib.Path.cwd()
    places = [here, here.parent, *here.glob("*"), *pathlib.Path("/content").glob("*"),
              *pathlib.Path("/content/drive/MyDrive").glob("*")]
    for p in places:
        if (p / "learning_agent" / "__init__.py").exists():
            return p
    return None

ROOT = find_root()
if ROOT is None:
    raise RuntimeError(
        "Can't find the `learning_agent` folder. Upload the whole project (zip it, upload, then run "
        "`!unzip -q -o learning-resource-agent.zip` and `%cd learning-resource-agent`), or clone your GitHub repo."
    )
sys.path.insert(0, str(ROOT))
os.environ.setdefault("DATA_DIR", str(ROOT / "data"))
print("Project folder:", ROOT)

if not os.getenv("GROQ_API_KEY"):
    try:
        from google.colab import userdata          # Colab secret named GROQ_API_KEY
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

In [ ]:
from learning_agent.config import config
from learning_agent.llm import configure_llama_index
config.groq_api_key = os.environ['GROQ_API_KEY']
config.data_dir = pathlib.Path(os.environ['DATA_DIR'])
configure_llama_index()
print('Model:', config.groq_model, '| YouTube enabled:', bool(config.youtube_api_key))

## Talk to the team
The first message goes to **InfoAgent**. If it asks a question, answer it by running the next cell again; once the profile is saved the agents hand off to each other automatically and you see their progress.

In [ ]:
from learning_agent.assistant import LearningAssistant
from IPython.display import Markdown, display

assistant = LearningAssistant('demo_user')

async def ask(message):
    res = await assistant.chat(message, on_status=print)   # prints agent activity live
    display(Markdown(res.reply))
    return res

res = await ask('I want to learn RAG with LlamaIndex. I know Python basics. Free resources only.')

In [ ]:
# If the agent asked a question, answer it here:
# res = await ask('I am a beginner with LLMs, about 5 hours per week.')

Results are saved to `data/resources.csv` and `data/ranked_resources.csv`. The structured result is in `res.result`.

In [ ]:
res.result['roadmap'] if res.done else 'Not finished yet'

## Gradio UI

In [ ]:
from learning_agent.app import build_app
build_app().launch()   # share=True for a public link